# Exercicis — la forma de cada model

**Optativa d'Aprenentatge automàtic — DAM/DAW 2n**

Heu vist cinc models, tots sobre Iris: k veïns, arbre, bosc, regressió
logística i SVM. Iris té 4 columnes; no es pot dibuixar. En aquest quadern
les dades són **fabricades**, amb `sklearn.datasets`, i tenen **només dues
columnes a posta**. Amb dues columnes es pot fer una cosa que amb Iris no
es podia: **dibuixar la frontera de decisió de cada model i veure quina
forma té**.

I la forma delata el model:

- **Regressió logística**: sempre una **recta**. Si les dades no se
  separen amb una recta, fracassa i es veu.
- **Arbre**: **escales**, rectangles amb els costats paral·lels als eixos.
  Cada tall és «aquesta columna és més gran que tal número».
- **Bosc**: les mateixes escales de l'arbre, suavitzades per la votació
  de molts arbres.
- **k-NN**: fronteres irregulars que s'enganxen als punts.
- **SVM (kernel `rbf`)**: corbes suaus.

Aquest és el quadern on es veu per fi en què es diferencien els cinc
models, perquè es dibuixen.

## Les eines que necessites

Aquest quadern demana aquestes eines. Estan llistades, no encadenades:
quina va abans de quina és la feina dels exercicis.

- De `sklearn.datasets`, per fabricar les dades: `make_blobs`,
  `make_moons`, `make_circles`, `make_classification`.
- De `sklearn.model_selection`: `train_test_split`.
- Els cinc models: `KNeighborsClassifier`, `DecisionTreeClassifier` (amb el
  seu `max_depth` i el seu `get_n_leaves()`), `RandomForestClassifier`,
  `LogisticRegression`, `SVC`.
- El mètode `.score()` de qualsevol model, per a la precisió.
- De pandas: `pd.DataFrame`.

Cada una està explicada a `EX_00_caixa_eines.ipynb`, amb la sintaxi, què li
has de donar, què et torna i el parany que té. Les dues funcions de dibuix
que necessites (`dibuixa_frontera` i `crea_models`) les tens fetes més
avall.

## Els objectes que manegues, i què els pots demanar

Les eines de la llista de dalt no treballen soltes: cadascuna et torna un
**objecte**, i el pas següent de l'exercici surt de demanar-li alguna cosa a aquell
objecte. Aquí tens, de cada un, el que faràs servir de veritat en aquest quadern:
els **atributs** (sense parèntesis) i els **mètodes** (amb parèntesis).

**El que tornen `make_blobs`, `make_moons`, `make_circles` i
`make_classification`** — no és un `Bunch` com els `load_*`: és una **tupla de dos
`ndarray`**, i per això s'escriu `Xb, yb = make_blobs(...)`. Si ho assignes a una
sola variable, tindràs la tupla sencera i el `.fit()` petarà.
- `X` és de forma `(n_samples, n_features)`: `(300, 2)` als quatre primers
  exercicis, `(300, 20)` a l'exercici 6
- `y` és de forma `(n_samples,)`: una etiqueta per fila

**L'`ndarray` de NumPy** — `Xb`, `Xm`, `Xc`, `Xcl`, i les seves particions:
- `.shape` — la tupla amb la forma. És el primer que has de mirar quan una funció de
  `sklearn.datasets` no et dona el que esperaves
- `.dtype` — el tipus dels valors que hi ha a dins
- `.T` — la matriu transposada
- `.min()` i `.max()` — els extrems, i és el que fa servir `dibuixa_frontera` per
  decidir on posa la graella
- `.mean(axis=0)` i `.std(axis=0)` — per columna; amb `axis=1`, per fila
- `.argmax()` — la **posició** del valor més gran, no el valor
- `.reshape()` — canvia la forma sense tocar els valors; és el que fa
  `dibuixa_frontera` amb la predicció de la graella
- `X[:, 0]` és la primera columna sencera i `X[:, 1]` la segona: és així com se li
  passen les dues coordenades a `ax.scatter`
- `model.predict(X) != y` et torna un array de booleans, i `.sum()` te'n compta els
  `True`: els errors

**Un model de scikit-learn** — els cinc que et torna `crea_models()`. Tots tenen
**la mateixa interfície**, i per això el bucle que els recorre pot ser el mateix per
a tots:
- `.fit(X, y)` — entrena amb aquestes dades; torna el model mateix
- `.predict(X)` — un array amb una etiqueta predita per fila. És l'únic que necessita
  `dibuixa_frontera`
- `.score(X, y)` — la precisió sobre aquestes dades, en un sol número
- `.classes_` — les classes que ha vist en entrenar, ordenades
- `.n_features_in_` — quantes columnes esperava; si li dones un nombre diferent a
  `.predict()`, peta
- `.get_n_leaves()` i `.get_depth()` — **només** l'arbre de decisió, i només després
  del `.fit()`. `max_depth` (sense guió baix al final) és el que li has **demanat**
  tu; `.get_depth()` és el que li ha **sortit** amb aquestes dades, i no tenen per
  què coincidir
- `.feature_importances_` — **només** l'arbre i el bosc: un pes per columna, que
  sumen 1
- `.coef_` i `.intercept_` — **només** `LogisticRegression`. Amb dues columnes,
  `.coef_` té dos números: són el pendent de la recta que dibuixa

**El diccionari de Python** — `crea_models()`, `apostes`, `precisions_blobs`,
`resultats_soroll`:
- `.items()` — parelles (clau, valor), que és com es recorre un joc de models
- `.keys()`, `.values()`, `.get(clau)`
- `apostes["moons"]["k-NN"]` baixa dos nivells; `precisions_blobs[nom] = model.score(...)`
  n'omple una casella

**El `DataFrame` de pandas** — `taula_soroll`:
- `.shape`, `.columns`, `.index`, `.dtypes` (atributs)
- `.T` — transposa files i columnes. És el que converteix
  `pd.DataFrame(resultats_soroll)` (models a les files) en la taula que et demana
  l'exercici 5 (`noise` a les files)
- `.round(3)`, `.sort_values(...)`, `.head()` (mètodes)

**Els objectes de matplotlib** — `plt.subplots(1, 5, figsize=(...))` et torna dues
coses: la `Figure` i un array d'`Axes`. Cada panell és un `Axes`, i és el que
`dibuixa_frontera` rep com a primer argument:
- `.contourf(xx, yy, Z, alpha=..., cmap=...)` — pinta la regió de cada classe
- `.scatter(x, y, c=..., cmap=..., edgecolors=..., s=...)` — els punts a sobre
- `.set_title(text, fontsize=...)`, `.set_xlabel(text)`, `.set_xticks([])`
- de la `Figure`: `.suptitle(text)`, `.tight_layout()`
- amb més d'un panell, `axs` és un array: `axs[0]`, o `axs.ravel()` per recórrer-lo
  amb un sol bucle. Amb `plt.subplots(1, 1)` **no** és un array, i indexar-lo peta

Dues regles que valen per a tot el quadern, perquè són les que fan petar el codi
sense que sembli que hi hagi res malament:

- **Un atribut va sense parèntesis i un mètode amb parèntesis.** `model.coef_` et
  dona els coeficients; `model.coef_()` dona `TypeError`, perquè no és una funció.
  I a l'inrevés: el `.describe` d'un DataFrame sense parèntesis no calcula res,
  t'ensenya el mètode.
- **El guió baix al final vol dir «això ho he après de les dades».** `.coef_`,
  `.feature_importances_` i `.classes_` **no existeixen abans de cridar `.fit()`**:
  si els demanes abans, salta un error. El mateix passa amb els atributs apresos
  de qualsevol transformador.

Per interrogar qualsevol d'aquests objectes pel teu compte —`type()`, `dir()`,
`objecte.metode?` i l'autocompletat amb TAB— tens
`FO_00_objectes_i_autocompletar.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

RANDOM_STATE = 42

## Dues eines fetes

Aquestes dues funcions es fan servir a tot el quadern. **No cal escriure-les
ni entendre-les línia a línia**; és per aquí per on comencen tots els
exercicis.

`dibuixa_frontera` fa una graella fina sobre el pla, pregunta al model de
quina classe és cada punt de la graella (`predict`) i pinta el resultat amb
`contourf`. Per sobre hi dibuixa les dades de veritat.

`crea_models` retorna els cinc models de sempre, sense entrenar, amb els
mateixos hiperparàmetres a tot arreu perquè les comparacions siguin justes.

In [ ]:
def dibuixa_frontera(ax, model, X, y, titol=""):
    """Pinta la regió que `model` assigna a cada classe, amb les dades a sobre."""
    marge = 0.5
    x1_min, x1_max = X[:, 0].min() - marge, X[:, 0].max() + marge
    x2_min, x2_max = X[:, 1].min() - marge, X[:, 1].max() + marge
    xx, yy = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                          np.linspace(x2_min, x2_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=15)
    ax.set_title(titol, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def crea_models():
    """Un joc nou dels cinc models, sempre amb els mateixos hiperparàmetres."""
    return {
        "k-NN": KNeighborsClassifier(n_neighbors=5),
        "Arbre": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Bosc": RandomForestClassifier(random_state=RANDOM_STATE),
        "Regressió logística": LogisticRegression(),
        "SVM (rbf)": SVC(kernel="rbf", random_state=RANDOM_STATE),
    }

## Exercici 1 — Fabricar i mirar

Genera tres conjunts de dades, tots amb dues columnes i `random_state=42`:

- `Xb, yb`: `make_blobs` amb `n_samples=300` i `centers=3`.
- `Xm, ym`: `make_moons` amb `n_samples=300` i `noise=0.2`.
- `Xc, yc`: `make_circles` amb `n_samples=300`, `noise=0.08` i `factor=0.4`.

Dibuixa'ls en una figura d'1 fila x 3 columnes (un `scatter` per conjunt,
pintat pel seu `y`).

Abans de continuar, **aposta**: per a cada conjunt i cada model dels cinc,
omple `apostes` amb `"bé"` o `"malament"` segons si creus que aquell model
hi farà bona feina. Fes-ho només mirant els dibuixos, sense entrenar res
encara — a l'exercici 7 es comprova.

**Com saps que ho has fet bé**: els tres dibuixos mostren formes clarament
diferents (grups separats, dues llunes entrellaçades, un cercle dins de
l'altre), i les 15 caselles de `apostes` tenen `"bé"` o `"malament"`, no
`None`.

In [ ]:
# Xb, yb = make_blobs(n_samples=300, centers=3, random_state=RANDOM_STATE)
# Xm, ym = make_moons(n_samples=300, noise=0.2, random_state=RANDOM_STATE)
# Xc, yc = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=RANDOM_STATE)

# figura 1x3 amb plt.subplots(1, 3, figsize=(...)): a cada Axes, un
# ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm") i un ax.set_title(...)

# aposta abans de veure cap número
apostes = {
    "blobs":   {"k-NN": None, "Arbre": None, "Bosc": None, "Regressió logística": None, "SVM (rbf)": None},
    "moons":   {"k-NN": None, "Arbre": None, "Bosc": None, "Regressió logística": None, "SVM (rbf)": None},
    "circles": {"k-NN": None, "Arbre": None, "Bosc": None, "Regressió logística": None, "SVM (rbf)": None},
}

## Exercici 2 — Blobs, el cas fàcil

Separa `Xb, yb` en entrenament i test (`test_size=0.3`,
`random_state=RANDOM_STATE`, `stratify=yb`). Entrena els cinc models de
`crea_models()` amb l'entrenament.

Dibuixa les cinc fronteres amb `dibuixa_frontera`, en una figura d'1x5, amb
la precisió de test a cada títol. Guarda les precisions en un diccionari
`precisions_blobs` (nom del model -> precisió).

**Com saps que ho has fet bé**: `precisions_blobs` ha de tenir les cinc
entrades i la figura els cinc panells, un per model, amb la precisió al
títol. El que importa aquí no és qui guanya, sinó **la forma** de cada
frontera: mira-te-les abans de llegir cap número i descriu cada una amb
paraules teves.

In [ ]:
# Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3,
#                                               random_state=RANDOM_STATE,
#                                               stratify=yb)

# recorre crea_models().items(): entrena cada model amb .fit(Xb_tr, yb_tr),
# guarda l'objecte entrenat en un diccionari i la seva precisio de test
# (.score(Xb_te, yb_te)) a precisions_blobs

# figura 1x5 amb plt.subplots(1, 5, figsize=(...)):
# dibuixa_frontera(ax, model, Xb_tr, yb_tr, titol=f"{nom}\n{precisio:.1%}")

precisions_blobs = {}  # nom del model -> precisió de test

## Exercici 3 — Moons, el cas que trenca la recta

Repeteix exactament l'exercici 2 però amb `Xm, ym` (les llunes). Guarda les
precisions a `precisions_moons`.

Després, en una cel·la Markdown, escriu dues o tres frases explicant **què
veus a la frontera de la regressió logística** que no veus a les altres
quatre, i per què passa (pensa en la forma de les dades i en la forma que
sap dibuixar cada model).

**Com saps que ho has fet bé**: el `train_test_split` ha de ser el mateix
que a l'exercici 2 (mateix `test_size` i mateix `random_state`), o les dues
taules de precisions no es poden comparar. Sobre l'explicació: cada cosa que
hi escriguis l'has de poder senyalar a la figura. Si dius que una frontera
té una forma determinada, assenyala-la; si dius que un model s'equivoca en
una zona concreta, comprova-ho amb `model.predict(Xm_te) != ym_te`.

In [ ]:
# igual que l'exercici 2 però amb Xm, ym: el mateix train_test_split
# (test_size=0.3, random_state=RANDOM_STATE, stratify=ym), un crea_models()
# nou, .fit() i .score() de cada model, i una figura 1x5 amb
# dibuixa_frontera

# anomena la particio Xm_tr, Xm_te, ym_tr, ym_te: l'exercici 4 la fa servir
# amb aquests noms

precisions_moons = {}  # nom del model -> precisió de test

*(Escriu aquí la teva explicació sobre la frontera de la regressió
logística a moons.)*

## Exercici 4 — L'escala de l'arbre

Amb `Xm_tr, ym_tr` (l'entrenament de moons de l'exercici 3), entrena quatre
`DecisionTreeClassifier` amb `max_depth` 1, 3, 5 i 20
(`random_state=RANDOM_STATE` a tots). Dibuixa només la frontera de
**l'arbre** (amb `dibuixa_frontera`) en una figura d'1x4, amb la
profunditat, el nombre de fulles (`model.get_n_leaves()`) i la precisió de
test a cada títol.

**Com saps que ho has fet bé**: amb `max_depth=1` l'arbre ha de tenir 2
fulles i la frontera ha de ser un únic tall recte. El nombre de fulles no
pot baixar mai quan puja la profunditat: si et baixa, has barrejat els
models. Apunta les quatre xifres de fulles i les quatre precisions de test
abans de treure'n cap conclusió, i compara com creix una sèrie i com creix
l'altra.

In [ ]:
# per a profunditat in [1, 3, 5, 20]:
#     entrena un DecisionTreeClassifier(max_depth=profunditat, random_state=RANDOM_STATE)
#     amb .fit(Xm_tr, ym_tr), i despres demana-li .get_n_leaves() i
#     .score(Xm_te, ym_te)

# figura 1x4 amb plt.subplots(1, 4, figsize=(...)):
#     dibuixa_frontera(ax, arbre, Xm_tr, ym_tr,
#              titol=f"profunditat={p}\n{n_fulles} fulles, {precisio:.1%}")

## Exercici 5 — Soroll

Genera tres versions de moons amb `noise` 0.05, 0.2 i 0.4 (300 mostres,
`random_state=RANDOM_STATE` a totes). Per a cada `noise`, separa en
entrenament/test, entrena els cinc models de `crea_models()` i guarda la
precisió de test de cadascun. Munta-ho en una taula (files = `noise`,
columnes = model) amb un `pd.DataFrame`.

**Com saps que ho has fet bé**: `taula_soroll` ha de tenir tres files (una
per `noise`) i cinc columnes (una per model), i cada cel·la ha de ser un
número entre 0 i 1. Genera els tres conjunts amb el mateix `random_state` i
la mateixa mida de mostra, o estaràs canviant dues coses alhora. Per
llegir-la, mira cada columna de dalt a baix, escriu quina direcció segueix,
i després mira si n'hi ha alguna que no segueixi la mateixa direcció que les
altres.

In [ ]:
resultats_soroll = {}  # noise -> {nom_model: precisio}

# per a noise in [0.05, 0.2, 0.4]:
#     genera make_moons(n_samples=300, noise=noise, random_state=RANDOM_STATE)
#     separa amb train_test_split(test_size=0.3, random_state=RANDOM_STATE,
#     stratify=y), i per a cada model de crea_models().items() fes .fit() i
#     guarda .score() del test

# taula_soroll = pd.DataFrame(resultats_soroll).T   (.T perque els noise
# quedin a les files i els models a les columnes)

## Exercici 6 — Fabricar un problema a mida

Genera `Xcl, ycl` amb `make_classification(n_samples=300, n_features=20,
n_informative=5, n_redundant=0, n_repeated=0, n_classes=2,
random_state=RANDOM_STATE)`: 20 columnes, de les quals només 5 porten
informació de veritat; les altres 15 són soroll. Amb 20 columnes ja no es
pot dibuixar cap frontera.

Separa en entrenament/test i compara un `DecisionTreeClassifier` sol
contra un `RandomForestClassifier`, tots dos amb `random_state=RANDOM_STATE`
i sense limitar la profunditat. Imprimeix les dues precisions de test.

**Com saps que ho has fet bé**: els dos models han de veure exactament la
mateixa partició: fes un sol `train_test_split` i passa'l als dos.
Imprimeix també la precisió d'entrenament de cadascun; si tots dos et donen
100 % a l'entrenament, la diferència que vegis a test ve de com
generalitzen, no de com s'ajusten. I abans de concloure res, repeteix-ho amb
tres o quatre `random_state` diferents al `train_test_split`: una diferència
que canvia de signe segons la partició no és una diferència.

In [ ]:
# Xcl, ycl = make_classification(n_samples=300, n_features=20, n_informative=5,
#                                 n_redundant=0, n_repeated=0, n_classes=2,
#                                 random_state=RANDOM_STATE)

# separa amb un sol train_test_split(Xcl, ycl, test_size=0.3,
# random_state=RANDOM_STATE, stratify=ycl) i passa la mateixa particio als dos

# arbre = DecisionTreeClassifier(random_state=RANDOM_STATE) -> .fit() -> .score()
# bosc  = RandomForestClassifier(random_state=RANDOM_STATE) -> .fit() -> .score()
# imprimeix tambe el .score() sobre l'entrenament de cadascun

## Exercici 7 — Tancament

Recupera les apostes de l'exercici 1. Per completar la comparació et falten
les precisions reals sobre `circles`: entrena els cinc models de
`crea_models()` sobre `Xc, yc` (mateix `train_test_split` que als exercicis
2 i 3) i guarda-les a `precisions_circles`.

Considera que una aposta era **encertada** si vas dir `"bé"` i la precisió
real és `>= 0.85`, o si vas dir `"malament"` i la precisió real és `< 0.85`.
Recorre `apostes`, `precisions_blobs`, `precisions_moons` i
`precisions_circles`, i imprimeix, per a cada conjunt i cada model, si
l'aposta va ser un encert o un error.

**Com saps que ho has fet bé**: has d'imprimir 15 línies, una per casella de
`apostes`, i cada una ha de dir el conjunt, el model, la teva aposta, la
precisió real i si va ser encert o error. Cap línia no pot sortir amb
`None`. El recompte d'encerts és el teu resultat, no el d'aquest enunciat:
de cada error que et surti, torna al dibuix del conjunt i a la frontera
d'aquell model i escriu què vas passar per alt.

In [ ]:
# Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3,
#                                               random_state=RANDOM_STATE,
#                                               stratify=yc)
# entrena els cinc models de crea_models() amb .fit() i omple
# precisions_circles amb el seu .score(Xc_te, yc_te)

precisions_circles = {}

LLINDAR = 0.85

# per a cada conjunt (blobs, moons, circles) i cada model (recorre
# apostes.items() i, a dins, el diccionari de cada conjunt amb .items()):
#     compara apostes[conjunt][model] amb la precisió real (>= LLINDAR -> "bé")
#     imprimeix si l'aposta va ser un encert o un error

## Resum

- Amb dues columnes es pot **dibuixar** la frontera de decisió: la forma
  delata el model (recta, escales, corbes, fronteres irregulars).
- La **regressió logística** només dibuixa rectes; falla clarament davant
  de dades que no se separen linealment (moons, circles).
- L'**arbre** dibuixa escales que es multipliquen amb la profunditat.
- El **bosc** suavitza les escales de l'arbre. Quant li guanya, i en quines
  dades, surt de les taules que has muntat als exercicis 5 i 6.
- El **soroll** de les dades entra dins de la comparació entre models:
  l'exercici 5 és el que et diu de quina manera.